# Duka Sales H1 2026

## Executive Summary
- Revenue was KES 71.8M from 1,000 orders in Jan-Jun 2026.
- Nairobi led with KES 26.1M (36%) from 376 orders.
- Laptops earned KES 41.5M (58%) while Tablets sold most units (691).
- Weekends had 277 orders averaging KES 71,972 vs 71,724 on weekdays.
- April peaked at KES 13.5M, 24% above June's KES 10.9M.

## Inspection

In [1]:
import pandas as pd
d=pd.read_csv("duka_sales.csv",parse_dates=["order_date"])
print(d.shape,d.dtypes,d.isna().sum().sum(),d.order_date.agg(["min","max"]),d.branch.unique(),d["channel"].unique(),sep="\n")

(1000, 7)
order_id               int64
order_date    datetime64[us]
branch                   str
product                  str
quantity               int64
channel                  str
unit_price             int64
dtype: object
0
min   2026-01-01
max   2026-06-29
Name: order_date, dtype: datetime64[us]
<StringArray>
['Nakuru', 'Nairobi', 'Mombasa', 'Eldoret', 'Kisumu']
Length: 5, dtype: str
<StringArray>
['In-store', 'Online', 'Phone order']
Length: 3, dtype: str


1,000 rows x 6; no nulls; 1 Jan-29 Jun 2026; 5 branches, 5 products, 3 channels.

## Derived columns

In [2]:
d["revenue"]=d.quantity*d.unit_price
d["month"]=d.order_date.dt.month_name();d["weekday"]=d.order_date.dt.day_name()
d["is_weekend"]=d.order_date.dt.dayofweek>=5
d["price_band"]=d.unit_price.ge(28000).map({True:"Premium",False:"Value"})
d.head(3)

C:\Users\Admin\AppData\Local\Temp\ipykernel_1928\3363493990.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d["is_weekend"]=d.order_date.dt.dayofweek>=5


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium


## Task 3: Branches

In [3]:
b=d.groupby("branch").agg(rev=("revenue","sum"),n=("order_id","count"))
b.assign(rr=b.rev.rank(ascending=False),nr=b.n.rank(ascending=False)).sort_values("rev",ascending=False)

,rev,n,rr,nr
branch,,,,
Nairobi,26106100,376,1.0,1.0
Mombasa,14003700,186,2.0,2.0
Nakuru,11970400,162,3.0,3.0
Kisumu,10868100,159,4.0,4.0
Eldoret,8844200,117,5.0,5.0


Rankings match: prices are identical across branches, so more orders mean more revenue. Nairobi has 36%.

## Task 4: Products

In [4]:
p=d.groupby("product").agg(rev=("revenue","sum"),u=("quantity","sum"))
display(p.nlargest(3,"rev"));p.nlargest(3,"u")

,rev,u
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


,rev,u
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


Laptop leads revenue (KES 41.5M) but Tablet leads units (691): Laptops cost KES 65,000 vs 28,000.

## Task 5: Weekend vs weekday

In [5]:
d.groupby("is_weekend").revenue.agg(["count","sum","mean"])

,count,sum,mean
is_weekend,,,
False,723,51856200,71723.651452
True,277,19936300,71972.202166


Weekend: 277 orders, KES 19.9M, avg 71,972. Weekday: 723, KES 51.9M, avg 71,724. Order value is equal.

## Task 6: Months

In [6]:
m=d.groupby(d.order_date.dt.month).revenue.sum()
print(m);(m.max()/m.min()-1)*100

order_date
1    11786600
2    11772000
3    12396800
4    13518300
5    11415400
6    10903400
Name: revenue, dtype: int64


np.float64(23.982427499679005)

Best April (KES 13.5M), worst June (KES 10.9M): April is 24.0% higher.

## Task 7: Premium online coast and capital

In [7]:
f=d[d.price_band.eq("Premium")&d.channel.eq("Online")&d.branch.isin(["Nairobi","Mombasa"])]
import os;os.makedirs("outputs",exist_ok=True)
f.to_csv("outputs/premium_online_coast_capital.csv",index=False)
len(f),f.revenue.sum()

(83, np.int64(10218000))

83 orders worth KES 10.2M.

## Task 8: Excel check

In [8]:
x=pd.read_excel("excel/duka_branch_summary.xlsx",nrows=5).set_index("Branch")["Revenue (SUMIFS)"]
(x.sort_index()==b.rev.sort_index()).all()

np.True_

Excel SUMIFS totals match pandas.